In [2]:
import pandas as pd
import numpy as np
import lifelines
from patsy import dmatrix
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from lifelines.utils import concordance_index
from lifelines import CoxPHFitter
from sklearn.metrics import log_loss
from sklearn.metrics import brier_score_loss
import matplotlib.cm as cm  
from scipy import stats
import matplotlib.ticker as mticker

In [3]:
df_train = pd.read_csv('df_train_cph5.csv')
df_test = pd.read_csv('df_test_cph5.csv') 

In [4]:
# partial log-likeliehood test for statification candidates

In [5]:
# select feature
feature = 'HIMCAREE'
penalizer = 0.01                          # choose small penalizer if necesary to prevenent multicolianrity
J = df_train[feature].nunique()           # number of strata (number fo different dataset values of that feature)

# step 1: identify constant columns in any stratum (we remove those featues later to prevent multicolinarity)
constant_cols_in_any_stratum = set()

for stratum in df_train[feature].unique():
    df_subset = df_train[df_train[feature] == stratum]
    constant_cols = df_subset.columns[df_subset.nunique() == 1].tolist()
    constant_cols_in_any_stratum.update(constant_cols) 

# convert to list for readability and print anme of columns
constant_cols_in_any_stratum = [col for col in constant_cols_in_any_stratum if col != feature]
if constant_cols_in_any_stratum:
    print(f"Columns that are constant in at least one stratum: {constant_cols_in_any_stratum}")

# step 2: remove identified constant columns
df_filtered = df_train.drop(columns=constant_cols_in_any_stratum)

# step 3: fit the stratified Cox model using the same set of features
formula="bs(BMICALC, degree=2, df=3) + bs(HRSLEEP, degree=1, df=2) + bs(STRONGFWK, degree=2, df=2) + bs(ALCDAYSYR, degree=1, df=2) + bs(ALCAMT, degree=2, df=2) + bs(POVERTY, degree=2, df=2) +" + " + ".join(df_filtered.columns.difference(['TIMETOEVENT', 'MORTSTAT','BMICALC', 'HRSLEEP', 'STRONGFWK', 'ALCDAYSYR', 'ALCAMT', 'POVERTY']+constant_cols_in_any_stratum+[feature]))
cph_stratified = CoxPHFitter(penalizer=penalizer)
cph_stratified.fit(df_filtered, duration_col='TIMETOEVENT', event_col='MORTSTAT', strata=[feature], formula=formula)
log_likelihood_stratified = cph_stratified.log_likelihood_

# fit separate Cox models for each stratum and sum log-likelihoods
log_likelihood_sum = 0
skipped_strata = 0
for stratum in df_filtered[feature].unique():
    print(f"\nProcessing stratum: {stratum}")

    df_subset = df_filtered[df_filtered[feature] == stratum].copy()
    df_subset = df_subset.drop(columns=[feature])  # drop the stratifying column

    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(df_subset, duration_col='TIMETOEVENT', event_col='MORTSTAT', formula=formula)
    log_likelihood_sum += cph.log_likelihood_

# compute the likelihood ratio test statistic (see defintion in the thesis)
chi2_stat = -2 * (log_likelihood_stratified - log_likelihood_sum)

# compute degrees of freedom
p = df_filtered.shape[1] + 7 - 2 
df_chi2 = (J - skipped_strata - 1) * p

# compute p-value using chi-square distribution
p_value = 1 - stats.chi2.cdf(chi2_stat, df_chi2)

# --- results ---
print(f"Stratified Model Log-Likelihood: {log_likelihood_stratified:.2f}")
print(f"\nLikelihood Ratio Test Statistic: {chi2_stat:.2f}")
print(f"Degrees of Freedom: {df_chi2}")
print(f"P-value: {p_value:.10f}")

# interpretation
if p_value < 0.05:
    print("Reject H0: Covariate effects are not the same across all strata.")
else:
    print("Fail to reject H0: Covariate effects are consistent across strata.")

Columns that are constant in at least one stratum: ['HINOTCOVE_CANCEREV', 'HINOTCOVE']

Processing stratum: 1.0

Processing stratum: 2.0
Stratified Model Log-Likelihood: -43588.70

Likelihood Ratio Test Statistic: 552.07
Degrees of Freedom: 56
P-value: 0.0000000000
Reject H0: Covariate effects are not the same across all strata.


In [6]:
# fit also startifed model and calcuate LL3Y and BS3Y

In [7]:
formula="bs(BMICALC, degree=2, df=3) + bs(HRSLEEP, degree=1, df=2) + bs(STRONGFWK, degree=2, df=2) + bs(ALCDAYSYR, degree=1, df=2) + bs(ALCAMT, degree=2, df=2) + bs(POVERTY, degree=2, df=2) +" + " + ".join(df_train.columns.difference(['TIMETOEVENT', 'MORTSTAT','BMICALC', 'HRSLEEP', 'STRONGFWK', 'ALCDAYSYR', 'ALCAMT', 'POVERTY']+[feature]))

cph_strt = CoxPHFitter(penalizer=0.000)
cph_strt.fit(df_train,
        duration_col='TIMETOEVENT', 
        event_col='MORTSTAT',
        strata=[feature],
        formula=formula)

<lifelines.CoxPHFitter: fitted with 82720 total observations, 78101 right-censored observations>

In [8]:
# predict the survival function for each individual
survival_probs_train_strt = cph_strt.predict_survival_function(df_train)
survival_probs_test_strt = cph_strt.predict_survival_function(df_test)

# extract 3y survival probabilities
survival_prob_train_strt = survival_probs_train_strt.iloc[3] 
survival_prob_test_strt = survival_probs_test_strt.iloc[3] 

# get 3-year event status based on TIMETOEVENT and MORTSTAT
status_3y_train = np.where((df_train["MORTSTAT"] == 1) & (df_train["TIMETOEVENT"] <= 3), 0, 1)
status_3y_test = np.where((df_test["MORTSTAT"] == 1) & (df_test["TIMETOEVENT"] <= 3), 0, 1)

print(f'CPH5 model startfied on feature {feature}')
print(f'APLL_is: {cph_strt.score(df_train, scoring_method="log_likelihood"):.4f}')
print(f'APLL_os: {cph_strt.score(df_test, scoring_method="log_likelihood"):.4f}')
print(f'LL3Y_is: {log_loss(status_3y_train, survival_prob_train_strt):.4f}')
print(f'LL3Y_os: {log_loss(status_3y_test, survival_prob_test_strt):.4f}')
print(f'BS3Y_is: {brier_score_loss(status_3y_train, survival_prob_train_strt):.4f}')
print(f'BS3Y_os: {brier_score_loss(status_3y_test, survival_prob_test_strt):.4f}')

CPH5 model startfied on feature HIMCAREE
APLL_is: -0.5201
APLL_os: -0.4306
LL3Y_is: 0.2141
LL3Y_os: 0.2119
BS3Y_is: 0.0412
BS3Y_os: 0.0408


In [9]:
# save for next model
df_train.to_csv("df_train_cph6.csv", index=False)
df_test.to_csv("df_test_cph6.csv", index=False)